# ResiDual CLAP — K-Fold Cross Validation

Questo notebook esegue la validazione k-fold per:
- **CLAP classico** (baseline zero-shot)
- **ResiDual CLAP** (con spectral reweighting ottimizzato)

sui dataset: **ESC-50**, **IRMAS**, **TinySOL**, **VocalSound**

---
**Pipeline per ogni fold:**
1. Split train / test
2. Valutazione baseline CLAP classico sul test set
3. *(solo ResiDual)* Fit PCA su `pca_fraction` del training set
4. *(solo ResiDual)* Ottimizzazione λ sul resto del training set
5. Valutazione ResiDual ottimizzato sul test set

## 0. Import e configurazione globale

In [1]:
import torch
import warnings
warnings.filterwarnings('ignore')

from CLAPWrapper import CLAPWrapper
from datasets.esc50     import ESC50
from datasets.irmas     import IRMAS
from datasets.tinysol   import TinySOL
from datasets.vocalsound import VocalSound
from models.residual_kfold import run_kfold, load_all_results, summarize_results

USE_CUDA = torch.cuda.is_available()
print(f'CUDA disponibile: {USE_CUDA}')

CUDA disponibile: False


## 1. Parametri globali

Modifica questi valori prima di eseguire i fold.

In [2]:
# ── Parametri k-fold ─────────────────────────────────────────────────────────
N_FOLDS            = 4      # numero di fold
PCA_FRACTION       = 0.15    # frazione del training set usata per fit PCA
PCA_SAMPLES        = 2000   # max campioni passati a fit_pca_on_data

# ── Parametri ottimizzazione λ ───────────────────────────────────────────────
MAX_EPOCHS         = 10
PATIENCE           = 3
LR                 = 1e-2

# ── Parametri ResiDual ───────────────────────────────────────────────────────
TARGET_LAYERS      = [2, 3]   # layer HTSAT su cui applicare RD*
VARIANCE_THRESHOLD = 0.95        # soglia PCA (0.95 = pca_95, 0.90 = pca_90)

# ── Parametri batch ──────────────────────────────────────────────────────────
BATCH_SIZE_AUDIO   = 1    # batch per fit_pca e optimize_lambda
BATCH_SIZE_EVAL    = 16   # batch per la valutazione

# ── Cartella di output ───────────────────────────────────────────────────────
SAVE_DIR = 'kfolds'

print('Parametri configurati.')

Parametri configurati.


## 2. Caricamento modello

Carichiamo **due** wrapper: uno classico e uno residual.  
Il wrapper residual viene ricreato prima di ogni fold per resettare i λ.

In [3]:
CLAP_VERSION  = '2023'              # '2022' o '2023'
CLAP_WEIGHTS  = None                # None = scarica automaticamente da HuggingFace

RESIDUAL_CONFIG = {
    'target_layers':      TARGET_LAYERS,
    'variance_threshold': VARIANCE_THRESHOLD,
}

# Wrapper classico (baseline)
classic_wrapper = CLAPWrapper(
    model_fp = CLAP_WEIGHTS,
    version  = CLAP_VERSION,
    use_cuda = USE_CUDA,
    type     = 'classic',
)

# Wrapper residual
residual_wrapper = CLAPWrapper(
    model_fp        = CLAP_WEIGHTS,
    version         = CLAP_VERSION,
    use_cuda        = USE_CUDA,
    type            = 'residual',
    residual_config = RESIDUAL_CONFIG,
)

print('Modelli caricati.')

Modelli caricati.


---
## 3. ESC-50

In [5]:
esc50 = ESC50(root='../data', download=True)
print(f'ESC-50: {len(esc50)} campioni, {len(esc50.classes)} classi')
print('Classi:', esc50.classes[:5], '...')

Dataset già presente in ../data/ESC-50-master, skip download.
Loading audio files


100%|██████████| 2000/2000 [00:00<00:00, 13196.79it/s]

✓ Cache di validazione trovata (2026-04-01T21:46:58.127961): 2000 validi, 0 corrotti. Skip validazione.
ESC-50: 2000 campioni, 50 classi
Classi: ['airplane', 'breathing', 'brushing teeth', 'can opening', 'car horn'] ...


In [6]:
# ── Baseline classica ────────────────────────────────────────────────────────
results_esc50_classic = run_kfold(
    dataset        = esc50,
    wrapper        = classic_wrapper,
    class_labels   = esc50.classes,
    n_folds        = N_FOLDS,
    model_type     = 'classic',
    batch_size_eval = BATCH_SIZE_EVAL,
    save_dir       = SAVE_DIR,
    dataset_name   = 'ESC-50',
)


  K-Fold (4 fold) — ESC-50 — tipo: classic


──────────────────────────────────────────────────
  FOLD 1 / 4
──────────────────────────────────────────────────
  [1/3] Baseline CLAP classico...


        Accuracy baseline: 0.9280
  Fold completato in 116.4s

──────────────────────────────────────────────────
  FOLD 2 / 4
──────────────────────────────────────────────────
  [1/3] Baseline CLAP classico...


        Accuracy baseline: 0.9380
  Fold completato in 101.9s

──────────────────────────────────────────────────
  FOLD 3 / 4
──────────────────────────────────────────────────
  [1/3] Baseline CLAP classico...


        Accuracy baseline: 0.9440
  Fold completato in 104.8s

──────────────────────────────────────────────────
  FOLD 4 / 4
──────────────────────────────────────────────────
  [1/3] Baseline CLAP classico...


        Accuracy baseline: 0.9440
  Fold completato in 107.5s

  Risultati salvati in: kfolds/ESC-50_classic_20260417_140303.json
  Baseline media:  0.9385 ± 0.0065



In [7]:
# ── ResiDual ─────────────────────────────────────────────────────────────────
results_esc50_residual = run_kfold(
    dataset             = esc50,
    wrapper             = residual_wrapper,
    class_labels        = esc50.classes,
    n_folds             = N_FOLDS,
    pca_fraction        = PCA_FRACTION,
    pca_samples         = PCA_SAMPLES,
    max_epochs          = MAX_EPOCHS,
    patience            = PATIENCE,
    lr                  = LR,
    variance_threshold  = VARIANCE_THRESHOLD,
    target_layers       = TARGET_LAYERS,
    model_type          = 'residual',
    batch_size_audio    = BATCH_SIZE_AUDIO,
    batch_size_eval     = BATCH_SIZE_EVAL,
    save_dir            = SAVE_DIR,
    dataset_name        = 'ESC-50',
)


  K-Fold (4 fold) — ESC-50 — tipo: residual


──────────────────────────────────────────────────
  FOLD 1 / 4
──────────────────────────────────────────────────
  [1/3] Baseline CLAP classico...


        Accuracy baseline: 0.9280
  [2/3] Fit PCA (225 campioni, max=2000)...


[ResiDual] Raccolta per PCA: 100%|██████████| 225/225 [01:05<00:00,  3.43it/s]


[ResiDual] Raccolti 225 campioni
  layer_2 | block 0 | head_0: k=14 | var@k=0.957
  layer_2 | block 0 | head_1: k=18 | var@k=0.958
  layer_2 | block 0 | head_2: k=16 | var@k=0.952
  layer_2 | block 0 | head_3: k=17 | var@k=0.961
  layer_2 | block 0 | head_4: k=14 | var@k=0.951
  layer_2 | block 0 | head_5: k=10 | var@k=0.950
  layer_2 | block 0 | head_6: k=11 | var@k=0.951
  layer_2 | block 0 | head_7: k=14 | var@k=0.957
  layer_2 | block 0 | head_8: k=18 | var@k=0.955
  layer_2 | block 0 | head_9: k=18 | var@k=0.952
  layer_2 | block 0 | head_10: k=16 | var@k=0.953
  layer_2 | block 0 | head_11: k=15 | var@k=0.953
  layer_2 | block 0 | head_12: k=14 | var@k=0.955
  layer_2 | block 0 | head_13: k=14 | var@k=0.954
  layer_2 | block 0 | head_14: k=18 | var@k=0.959
  layer_2 | block 0 | head_15: k=16 | var@k=0.960
  layer_2 | block 1 | head_0: k=17 | var@k=0.959
  layer_2 | block 1 | head_1: k=18 | var@k=0.953
  layer_2 | block 1 | head_2: k=19 | var@k=0.950
  layer_2 | block 1 | head_3: 

KeyboardInterrupt: 

---
## 4. IRMAS

In [ ]:
irmas = IRMAS(root='../data', download=True)
print(f'IRMAS: {len(irmas)} campioni, {len(irmas.classes)} classi')

In [ ]:
results_irmas_classic = run_kfold(
    dataset        = irmas,
    wrapper        = classic_wrapper,
    class_labels   = irmas.classes,
    n_folds        = N_FOLDS,
    model_type     = 'classic',
    batch_size_eval = BATCH_SIZE_EVAL,
    save_dir       = SAVE_DIR,
    dataset_name   = 'IRMAS',
)

In [ ]:
results_irmas_residual = run_kfold(
    dataset             = irmas,
    wrapper             = residual_wrapper,
    class_labels        = irmas.classes,
    n_folds             = N_FOLDS,
    pca_fraction        = PCA_FRACTION,
    pca_samples         = PCA_SAMPLES,
    max_epochs          = MAX_EPOCHS,
    patience            = PATIENCE,
    lr                  = LR,
    variance_threshold  = VARIANCE_THRESHOLD,
    target_layers       = TARGET_LAYERS,
    model_type          = 'residual',
    batch_size_audio    = BATCH_SIZE_AUDIO,
    batch_size_eval     = BATCH_SIZE_EVAL,
    save_dir            = SAVE_DIR,
    dataset_name        = 'IRMAS',
)

---
## 5. TinySOL

In [ ]:
tinysol = TinySOL(root='../data', download=True)
print(f'TinySOL: {len(tinysol)} campioni, {len(tinysol.classes)} classi')

In [ ]:
results_tinysol_classic = run_kfold(
    dataset        = tinysol,
    wrapper        = classic_wrapper,
    class_labels   = tinysol.classes,
    n_folds        = N_FOLDS,
    model_type     = 'classic',
    batch_size_eval = BATCH_SIZE_EVAL,
    save_dir       = SAVE_DIR,
    dataset_name   = 'TinySOL',
)

In [ ]:
results_tinysol_residual = run_kfold(
    dataset             = tinysol,
    wrapper             = residual_wrapper,
    class_labels        = tinysol.classes,
    n_folds             = N_FOLDS,
    pca_fraction        = PCA_FRACTION,
    pca_samples         = PCA_SAMPLES,
    max_epochs          = MAX_EPOCHS,
    patience            = PATIENCE,
    lr                  = LR,
    variance_threshold  = VARIANCE_THRESHOLD,
    target_layers       = TARGET_LAYERS,
    model_type          = 'residual',
    batch_size_audio    = BATCH_SIZE_AUDIO,
    batch_size_eval     = BATCH_SIZE_EVAL,
    save_dir            = SAVE_DIR,
    dataset_name        = 'TinySOL',
)

---
## 6. VocalSound

In [ ]:
vocalsound = VocalSound(root='../data', download=True)
print(f'VocalSound: {len(vocalsound)} campioni, {len(vocalsound.classes)} classi')

In [ ]:
results_vocal_classic = run_kfold(
    dataset        = vocalsound,
    wrapper        = classic_wrapper,
    class_labels   = vocalsound.classes,
    n_folds        = N_FOLDS,
    model_type     = 'classic',
    batch_size_eval = BATCH_SIZE_EVAL,
    save_dir       = SAVE_DIR,
    dataset_name   = 'VocalSound',
)

In [ ]:
results_vocal_residual = run_kfold(
    dataset             = vocalsound,
    wrapper             = residual_wrapper,
    class_labels        = vocalsound.classes,
    n_folds             = N_FOLDS,
    pca_fraction        = PCA_FRACTION,
    pca_samples         = PCA_SAMPLES,
    max_epochs          = MAX_EPOCHS,
    patience            = PATIENCE,
    lr                  = LR,
    variance_threshold  = VARIANCE_THRESHOLD,
    target_layers       = TARGET_LAYERS,
    model_type          = 'residual',
    batch_size_audio    = BATCH_SIZE_AUDIO,
    batch_size_eval     = BATCH_SIZE_EVAL,
    save_dir            = SAVE_DIR,
    dataset_name        = 'VocalSound',
)

---
# 7. Analisi e Visualizzazione Risultati

Questa sezione carica **tutti** i file JSON nella cartella `kfolds/` e produce i plot comparativi.  
Puoi eseguirla anche standalone dopo aver già eseguito i fold in sessioni precedenti.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from models.residual_kfold import load_all_results

# ── Carica tutti i risultati ──────────────────────────────────────────────────
all_results = load_all_results('kfolds')
print(f'File JSON trovati: {len(all_results)}')
for r in all_results:
    print(f"  {r['dataset']:15s} | {r['model_type']:8s} | "
          f"baseline={r['mean_baseline_acc']:.4f}", end='')
    if 'mean_residual_acc' in r:
        print(f" | residual={r['mean_residual_acc']:.4f}", end='')
    print()

In [ ]:
# ── Costruisce dizionario aggregato ──────────────────────────────────────────
# Struttura: {dataset: {'classic': {mean, std, folds}, 'residual': {mean, std, folds}}}

agg = {}
for r in all_results:
    ds    = r['dataset']
    mtype = r['model_type']
    if ds not in agg:
        agg[ds] = {}

    # Accuracy per fold
    baseline_per_fold = [f['baseline_acc'] for f in r['folds']]

    # La baseline classica è la stessa nei run 'classic' e 'residual'
    # Prendiamo sempre quella del run 'classic' se presente, altrimenti quella del run corrente
    if mtype == 'classic' or 'classic' not in agg[ds]:
        agg[ds]['classic'] = {
            'mean':  r['mean_baseline_acc'],
            'std':   r['std_baseline_acc'],
            'folds': baseline_per_fold,
        }

    if mtype == 'residual' and 'mean_residual_acc' in r:
        residual_per_fold = [f.get('residual_acc', 0.0) for f in r['folds']]
        agg[ds]['residual'] = {
            'mean':  r['mean_residual_acc'],
            'std':   r['std_residual_acc'],
            'folds': residual_per_fold,
        }

datasets = sorted(agg.keys())
print('Dataset trovati:', datasets)

### 7.1 Bar chart — Accuracy media per dataset

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x      = np.arange(len(datasets))
width  = 0.35
color_classic  = '#4C72B0'
color_residual = '#DD8452'

classic_means  = [agg[d]['classic']['mean']  for d in datasets]
classic_stds   = [agg[d]['classic']['std']   for d in datasets]
residual_means = [agg[d].get('residual', {}).get('mean', 0) for d in datasets]
residual_stds  = [agg[d].get('residual', {}).get('std',  0) for d in datasets]

bars1 = ax.bar(x - width/2, classic_means,  width,
               yerr=classic_stds,  capsize=5,
               color=color_classic,  label='CLAP Classico', alpha=0.85)
bars2 = ax.bar(x + width/2, residual_means, width,
               yerr=residual_stds, capsize=5,
               color=color_residual, label='ResiDual CLAP', alpha=0.85)

# Annotazioni valore
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(datasets, fontsize=11)
ax.set_ylabel('Accuracy (media k-fold)', fontsize=11)
ax.set_title('Zero-Shot Accuracy — CLAP Classico vs ResiDual', fontsize=13, fontweight='bold')
ax.set_ylim(0, min(1.0, max(classic_means + residual_means) + 0.15))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('kfolds/bar_accuracy.png', dpi=150)
plt.show()
print('Salvato: kfolds/bar_accuracy.png')

### 7.2 Box plot — Distribuzione accuracy per fold

In [ ]:
n_ds   = len(datasets)
fig, axes = plt.subplots(1, n_ds, figsize=(4 * n_ds, 5), sharey=True)

if n_ds == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    data_to_plot = []
    labels_box   = []
    colors_box   = []

    if 'classic' in agg[ds] and agg[ds]['classic']['folds']:
        data_to_plot.append(agg[ds]['classic']['folds'])
        labels_box.append('Classic')
        colors_box.append(color_classic)

    if 'residual' in agg[ds] and agg[ds]['residual']['folds']:
        data_to_plot.append(agg[ds]['residual']['folds'])
        labels_box.append('ResiDual')
        colors_box.append(color_residual)

    bp = ax.boxplot(data_to_plot, patch_artist=True, widths=0.4,
                    medianprops=dict(color='black', linewidth=2))

    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    # Sovrappone i punti dei singoli fold
    for i, (data, color) in enumerate(zip(data_to_plot, colors_box), start=1):
        jitter = np.random.uniform(-0.08, 0.08, len(data))
        ax.scatter([i + j for j in jitter], data, color=color,
                   s=30, zorder=5, alpha=0.8)

    ax.set_xticks(range(1, len(labels_box) + 1))
    ax.set_xticklabels(labels_box, fontsize=10)
    ax.set_title(ds, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Accuracy per fold', fontsize=11)
fig.suptitle('Distribuzione Accuracy — K-Fold', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('kfolds/boxplot_folds.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvato: kfolds/boxplot_folds.png')

### 7.3 Delta plot — Guadagno ResiDual vs Classico

In [ ]:
deltas = []
ds_with_residual = []

for ds in datasets:
    if 'residual' in agg[ds] and 'classic' in agg[ds]:
        delta = agg[ds]['residual']['mean'] - agg[ds]['classic']['mean']
        deltas.append(delta)
        ds_with_residual.append(ds)

if deltas:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors  = [color_residual if d > 0 else '#C44E52' for d in deltas]

    bars = ax.barh(ds_with_residual, deltas, color=colors, alpha=0.85, height=0.5)

    for bar, delta in zip(bars, deltas):
        x_pos = bar.get_width() + (0.001 if delta >= 0 else -0.001)
        ha    = 'left' if delta >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2,
                f'{delta:+.4f}', va='center', ha=ha, fontsize=9)

    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Δ Accuracy (ResiDual − Classico)', fontsize=11)
    ax.set_title('Guadagno ResiDual rispetto al Baseline', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('kfolds/delta_accuracy.png', dpi=150)
    plt.show()
    print('Salvato: kfolds/delta_accuracy.png')
else:
    print('Nessun risultato residual disponibile per il delta plot.')

### 7.4 Spider / Radar chart — Confronto su tutti i dataset

In [ ]:
ds_radar = [d for d in datasets if 'classic' in agg[d]]

if len(ds_radar) >= 3:
    N      = len(ds_radar)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]   # chiude il poligono

    classic_vals  = [agg[d]['classic']['mean']               for d in ds_radar] + \
                    [agg[ds_radar[0]]['classic']['mean']]
    residual_vals = [agg[d].get('residual', {}).get('mean', 0) for d in ds_radar] + \
                    [agg[ds_radar[0]].get('residual', {}).get('mean', 0)]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    # Griglia e tick
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(ds_radar, fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='grey')
    ax.grid(color='grey', alpha=0.3)

    # Classico
    ax.plot(angles, classic_vals, color=color_classic, linewidth=2, label='CLAP Classico')
    ax.fill(angles, classic_vals, color=color_classic, alpha=0.15)

    # ResiDual
    if any(v > 0 for v in residual_vals):
        ax.plot(angles, residual_vals, color=color_residual, linewidth=2, label='ResiDual CLAP')
        ax.fill(angles, residual_vals, color=color_residual, alpha=0.15)

    ax.set_title('Spider Chart — Accuracy per Dataset', fontsize=13,
                 fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

    plt.tight_layout()
    plt.savefig('kfolds/spider_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Salvato: kfolds/spider_accuracy.png')
else:
    print(f'Spider chart richiede almeno 3 dataset (trovati: {len(ds_radar)}).')

### 7.5 Tabella riepilogativa

In [ ]:
print(f"{'Dataset':<15} {'Classic (mean±std)':<22} {'ResiDual (mean±std)':<22} {'Delta':>8}")
print('─' * 70)

for ds in datasets:
    c = agg[ds].get('classic', {})
    r = agg[ds].get('residual', {})

    c_str = f"{c.get('mean', 0):.4f} ± {c.get('std', 0):.4f}" if c else '   —   '
    r_str = f"{r.get('mean', 0):.4f} ± {r.get('std', 0):.4f}" if r else '   —   '

    if c and r:
        delta = r['mean'] - c['mean']
        d_str = f'{delta:+.4f}'
    else:
        d_str = '   —'

    print(f"{ds:<15} {c_str:<22} {r_str:<22} {d_str:>8}")